# Decision Logic

In [ ]:
import numpy as np
from laboneq._automation.utils.yml_parser import load_automation_parameters
from laboneq.simple import *

from laboneq_applications._automation import WorkflowAutomation, WorkflowLayer
from laboneq_applications._automation.workflow.workflow_logic import (
    FixedParameterUpdate,
)
from laboneq_applications.experiments import (
    qubit_spectroscopy,
)
from laboneq_applications.qpu_types.tunable_transmon import demo_platform

# Create a demonstration QuantumPlatform for a 4-qubit tunable-transmon QPU:
qt_platform = demo_platform(n_qubits=4)

# The platform contains a setup, which is an ordinary LabOne Q DeviceSetup (1x PQSC, 1x SHFQC, 1x HDAWG):
setup = qt_platform.setup

# And a 4-qubit tunable-transmon QPU:
qpu = qt_platform.qpu

# Inside the QPU, we have quantum elements, which is a list of four LabOne Q Application
# Library TunableTransmonQubit qubits:
qubits = qpu.quantum_elements

# We connect to the session in emulation mode:
session = Session(setup)
session.connect(do_emulation=True)

## Run the experiment in isolation

In [ ]:
from laboneq_applications.experiments.options import TuneUpWorkflowOptions

opts = TuneUpWorkflowOptions()
opts.evaluate = True
opts.update = True
opts

In [ ]:
experiment_workflow = qubit_spectroscopy.experiment_workflow(
    session=session,
    qpu=qpu,
    qubits=qubits[0],
    frequencies=np.linspace(6e9, 6.3e9, 11),
    options=opts,
)

workflow_result = experiment_workflow.run()

In [ ]:
experiment_workflow.graph.tree

In [ ]:
workflow_result.output.data.q0.result

## Run the experiment in the automation (using the run method)

In [ ]:
auto = WorkflowAutomation(
    session,
    qpu=qpu,
    automation_parameters=load_automation_parameters("decision_logic.yml"),
)

qs1 = WorkflowLayer(
    qubit_spectroscopy.experiment_workflow,
    ["q0", "q1", "q2"],
    key="qs1",
    depends_on=["__root__"],
)
qs2 = WorkflowLayer(
    qubit_spectroscopy.experiment_workflow,
    ["q0", "q1", "q2"],
    key="qs2",
    depends_on=["qs1"],
)
qs3 = WorkflowLayer(
    qubit_spectroscopy.experiment_workflow,
    ["q0", "q1", "q2"],
    key="qs3",
    depends_on=["qs2"],
)

auto.add_layer(qs1)
auto.add_layer(qs2)
auto.add_layer(qs3)
auto.plot();

In [ ]:
auto.run()
auto.plot();

In [ ]:
auto.get_layer("qs2")

## Run the experiment in the automation (using the run_layer method)

In [ ]:
auto = WorkflowAutomation(
    session,
    qpu=qpu,
    automation_parameters=load_automation_parameters("decision_logic.yml"),
)

qs1 = WorkflowLayer(
    qubit_spectroscopy.experiment_workflow,
    ["q0", "q1", "q2"],
    key="qs1",
    depends_on=["__root__"],
)
qs2 = WorkflowLayer(
    qubit_spectroscopy.experiment_workflow,
    ["q0", "q1", "q2"],
    key="qs2",
    depends_on=["qs1"],
)
qs3 = WorkflowLayer(
    qubit_spectroscopy.experiment_workflow,
    ["q0", "q1", "q2"],
    key="qs3",
    depends_on=["qs2"],
)

auto.add_layer(qs1)
auto.add_layer(qs2)
auto.add_layer(qs3)
auto.plot();

In [ ]:
eval_output, new_layer_key, new_params = auto.run_layer("qs1")
print(eval_output)
print(new_layer_key)
print(new_params)
auto.plot();

In [ ]:
eval_output, new_layer_key, new_params = auto.run_layer("qs2")
print(eval_output)
print(new_layer_key)
print(new_params)
auto.plot();

In [ ]:
eval_output, new_layer_key, new_params = auto.run_layer("qs3")
print(eval_output)
print(new_layer_key)
print(new_params)
auto.plot();

## Run the experiment logic in-line

In [ ]:
auto = WorkflowAutomation(
    session,
    qpu=qpu,
    automation_parameters=load_automation_parameters("decision_logic.yml"),
)

qs1 = WorkflowLayer(
    qubit_spectroscopy.experiment_workflow,
    ["q0", "q1", "q2"],
    key="qs1",
    depends_on=["__root__"],
)
qs2 = WorkflowLayer(
    qubit_spectroscopy.experiment_workflow,
    ["q0", "q1", "q2"],
    key="qs2",
    depends_on=["qs1"],
)
qs3 = WorkflowLayer(
    qubit_spectroscopy.experiment_workflow,
    ["q0", "q1", "q2"],
    key="qs3",
    depends_on=["qs2"],
)

auto.add_layer(qs1)
auto.add_layer(qs2)
auto.add_layer(qs3)
auto.plot();

In [ ]:
auto.run_layer("qs1")
auto.plot();

In [ ]:
auto.run_layer("qs2")
auto.plot();

In [ ]:
auto.run_layer(
    "qs3",
    logic=FixedParameterUpdate(
        "qs3", {"q2": {"evaluation_fit_r2_thresholds": -0.0001}}
    ),
)
auto.plot();